## CP.2.2.1.b2c, Sauer3

Use code fragments for Gaussian elimination in the previous section to write a Python script to take a matrix $A$ as input and output $L$ and $U$. **No row exchanges are allowed** — the program should be designed to shut down if it encounters a zero pivot. Check your program by factoring the matrices in EX.2.2.2.b2c.

**(b)**

$$
A = \begin{pmatrix} 4 & 2 & 0 \\ 4 & 4 & 2 \\ 2 & 2 & 3 \end{pmatrix}
$$

**(c)**

$$
A = \begin{pmatrix} 1 & -1 & 1 & 2 \\ 0 & 2 & 1 & 0 \\ 1 & 3 & 4 & 4 \\ 0 & 2 & 1 & -1 \end{pmatrix}
$$

In [13]:
import numpy as np
from math import sqrt
from scipy.optimize import fsolve

In [8]:
def lu_factor(A):
    A = np.array(A, dtype=float)
    n = A.shape[0]
    L = np.eye(n)
    U = A.copy()

    for col in range(n - 1):
        pivot = U[col, col]
        if abs(pivot) < 1e-14:
            raise ValueError(f"Zero pivot encountered at position ({col}, {col}). "
                              f"No row exchanges allowed -- shutting down.")
        for row in range(col + 1, n):
            m = U[row, col] / pivot       # multiplier
            L[row, col] = m               # store multiplier in L
            U[row, :] = U[row, :] - m * U[col, :]   # eliminate entry

    return L, U


# test matrix b
A_b = [[4, 2, 0],
       [4, 4, 2],
       [2, 2, 3]]

L_b, U_b = lu_factor(A_b)
print("Matrix (b):")
print("L =\n", L_b)
print("U =\n", U_b)
print("Check L @ U =\n", L_b @ U_b)
print()

# test matrix c
A_c = [[1, -1, 1, 2],
       [0, 2, 1, 0],
       [1, 3, 4, 4],
       [0, 2, 1, -1]]

L_c, U_c = lu_factor(A_c)
print("Matrix (c):")
print("L =\n", L_c)
print("U =\n", U_c)
print("Check L @ U =\n", L_c @ U_c)

Matrix (b):
L =
 [[1.  0.  0. ]
 [1.  1.  0. ]
 [0.5 0.5 1. ]]
U =
 [[4. 2. 0.]
 [0. 2. 2.]
 [0. 0. 2.]]
Check L @ U =
 [[4. 2. 0.]
 [4. 4. 2.]
 [2. 2. 3.]]

Matrix (c):
L =
 [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [1. 2. 1. 0.]
 [0. 1. 0. 1.]]
U =
 [[ 1. -1.  1.  2.]
 [ 0.  2.  1.  0.]
 [ 0.  0.  1.  2.]
 [ 0.  0.  0. -1.]]
Check L @ U =
 [[ 1. -1.  1.  2.]
 [ 0.  2.  1.  0.]
 [ 1.  3.  4.  4.]
 [ 0.  2.  1. -1.]]


## CP.2.2.2.b, Sauer3

Add two-step back substitution to your script from CP.2.2.1, and use it to solve the systems in EX.2.2.4.b.

**(b)**

$$
\begin{pmatrix} 4 & 2 & 0 \\ 4 & 4 & 2 \\ 2 & 2 & 3 \end{pmatrix}
\begin{pmatrix} x_0 \\ x_1 \\ x_2 \end{pmatrix}
=
\begin{pmatrix} 2 \\ 4 \\ 6 \end{pmatrix}
$$

In [10]:
def forward_substitution(L, b):
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - L[i, :i] @ y[:i]
    return y


def back_substitution(U, y):
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x


def lu_solve(A, b):
    L, U = lu_factor(A)
    y = forward_substitution(L, b)   # solve Ly = b
    x = back_substitution(U, y)      # solve Ux = y
    return x, L, U, y


# solve system b
A_b = [[4, 2, 0],
       [4, 4, 2],
       [2, 2, 3]]
b_b = [2, 4, 6]

x_b, L_b, U_b, y_b = lu_solve(A_b, b_b)

print("Matrix (b):")
print("L =\n", L_b)
print("U =\n", U_b)
print("y (from Ly = b) =", y_b)
print("x (solution, from Ux = y) =", x_b)
print("Check A @ x =", np.array(A_b) @ x_b, " (should equal b =", b_b, ")")

Matrix (b):
L =
 [[1.  0.  0. ]
 [1.  1.  0. ]
 [0.5 0.5 1. ]]
U =
 [[4. 2. 0.]
 [0. 2. 2.]
 [0. 0. 2.]]
y (from Ly = b) = [2. 2. 4.]
x (solution, from Ux = y) = [ 1. -1.  2.]
Check A @ x = [2. 4. 6.]  (should equal b = [2, 4, 6] )


## EX.2.3.6, Sauer3

Find the relative forward and backward errors and error magnification factor for the following approximate solutions of the system

$$
\begin{cases} x_1 + 2x_2 = 3 \\ 2x_1 + 4.01x_2 = 6.01 \end{cases}
$$

**(a)** $\begin{pmatrix} -10 \\ 6 \end{pmatrix}$

**(b)** $\begin{pmatrix} -100 \\ 52 \end{pmatrix}$

**(c)** $\begin{pmatrix} -600 \\ 301 \end{pmatrix}$

**(d)** $\begin{pmatrix} -599 \\ 301 \end{pmatrix}$

**(e)** What is the condition number of the coefficient matrix?

In [11]:

# coefficient matrix and right-hand side
A = np.array([[1, 2],
              [2, 4.01]])
b = np.array([3, 6.01])

# exact solution
x_exact = np.linalg.solve(A, b)
print("Exact solution x* =", x_exact)
print()

# approximate solutions given in the problem
approx_solutions = {
    'a': np.array([-10, 6]),
    'b': np.array([-100, 52]),
    'c': np.array([-600, 301]),
    'd': np.array([-599, 301]),
}

# use the infinity norm (standard choice for this textbook)
norm = lambda v: np.linalg.norm(v, ord=np.inf)

print(f"{'Case':<6}{'Forward rel. err.':<22}{'Backward rel. err.':<22}{'Magnification factor':<22}")
results = {}
for label, x_approx in approx_solutions.items():
    # forward error: difference between approximate and exact solution
    forward_err = norm(x_approx - x_exact) / norm(x_exact)

    # residual: how well does x_approx satisfy Ax = b?
    residual = b - A @ x_approx

    # backward error: relative residual size
    backward_err = norm(residual) / (norm(A) * norm(x_approx))

    # rrror magnification factor = forward error / backward error
    magnification = forward_err / backward_err

    results[label] = (forward_err, backward_err, magnification)
    print(f"{label:<6}{forward_err:<22.6e}{backward_err:<22.6e}{magnification:<22.4f}")

print()

# e
cond_A = np.linalg.cond(A, np.inf)
print("Condition number (infinity norm) of A:", cond_A)

Exact solution x* = [1. 1.]

Case  Forward rel. err.     Backward rel. err.    Magnification factor  
a     1.100000e+01          3.244592e-02          339.0256              
b     1.010000e+02          4.176373e-03          24183.6653            
c     6.010000e+02          2.773156e-04          2167206.0000          
d     6.000000e+02          8.333356e-04          719998.0000           

Condition number (infinity norm) of A: 3612.0100000000766


## CP.2.3.4, Sauer3

Let $A$ be the $n$-by-$n$ matrix with entries

$$
a_{ij} = \sqrt{(i-j)^2 + \frac{n}{10}}.
$$

Define $x = (1, \dots, 1)^T$ and $b = Ax$. For $n = 100, 200, 300, 400,$ and $500$, use the Python program from Computer Problem 2.1.1 or NumPy's `numpy.linalg.solve` command to compute $x_c$, the double precision computed solution. Calculate the infinity norm of the forward error for each solution. Find the five error magnification factors of the problems $Ax = b$, and compare with the corresponding condition numbers.

**Hint:** Note that this $a_{ij} = \sqrt{(i-j)^2 + \dfrac{n}{10}}$ formula is the same with 0-base indexing (Python) or 1-base indexing (Sauer and MATLAB). Here is a code snippet to generate $A$ with $n = 5$:

```python
from math import sqrt
n = 5
A = np.zeros([n, n], dtype=float)
for i in range(0, n):
    for j in range(0, n):
        A[i, j] = sqrt((i - j)**2 + n / 10.)
print(A)
```

```
[[0.70710678 1.22474487 2.12132034 3.082207   4.0620192 ]
 [1.22474487 0.70710678 1.22474487 2.12132034 3.082207  ]
 [2.12132034 1.22474487 0.70710678 1.22474487 2.12132034]
 [3.082207   2.12132034 1.22474487 0.70710678 1.22474487]
 [4.0620192  3.082207   2.12132034 1.22474487 0.70710678]]
```

In [12]:
def build_A(n):
    A = np.zeros([n, n], dtype=float)
    for i in range(n):
        for j in range(n):
            A[i, j] = sqrt((i - j)**2 + n / 10.)
    return A

ns = [100, 200, 300, 400, 500]

print(f"{'n':<6}{'fwd error (inf norm)':<24}{'magnification factor':<24}{'cond_inf(A)':<20}")

results = []
for n in ns:
    A = build_A(n)
    x_exact = np.ones(n)
    b = A @ x_exact

    x_computed = np.linalg.solve(A, b) # double precision computed solution

    # forward error
    forward_err = np.linalg.norm(x_computed - x_exact, ord=np.inf) / np.linalg.norm(x_exact, ord=np.inf)

    # backward error
    residual = b - A @ x_computed
    backward_err = np.linalg.norm(residual, ord=np.inf) / (np.linalg.norm(A, ord=np.inf) * np.linalg.norm(x_computed, ord=np.inf))

    # error magnification factor
    magnification = forward_err / backward_err

    # condition number
    cond_A = np.linalg.cond(A, np.inf)

    results.append((n, forward_err, backward_err, magnification, cond_A))
    print(f"{n:<6}{forward_err:<24.6e}{magnification:<24.6e}{cond_A:<20.6e}")

print()
print("Full comparison table:")
print(f"{'n':<6}{'forward err':<16}{'backward err':<16}{'magnification':<18}{'cond(A)':<16}")
for n, fwd, bwd, mag, cond in results:
    print(f"{n:<6}{fwd:<16.4e}{bwd:<16.4e}{mag:<18.4e}{cond:<16.4e}")

n     fwd error (inf norm)    magnification factor    cond_inf(A)         
100   3.847440e-09            1.052262e+07            6.184308e+07        
200   8.304494e-07            1.518181e+09            1.290662e+10        
300   2.809067e-05            4.336784e+10            6.203932e+11        
400   8.341791e-04            9.169224e+11            1.475285e+13        
500   1.159182e-02            1.677238e+13            2.289323e+14        

Full comparison table:
n     forward err     backward err    magnification     cond(A)         
100   3.8474e-09      3.6564e-16      1.0523e+07        6.1843e+07      
200   8.3045e-07      5.4700e-16      1.5182e+09        1.2907e+10      
300   2.8091e-05      6.4773e-16      4.3368e+10        6.2039e+11      
400   8.3418e-04      9.0976e-16      9.1692e+11        1.4753e+13      
500   1.1592e-02      6.9113e-16      1.6772e+13        2.2893e+14      


## CP.2.7.1.b2c, Sauer3

Implement Newton's Method with appropriate starting points to find all solutions. Check with EX.2.7.3 to make sure your answers are correct.

**(b)**

$$
\begin{cases} u^2 + 4v^2 = 4 \\ 4u^2 + v^2 = 4 \end{cases}
$$

**(c)**

$$
\begin{cases} u^2 - 4v^2 = 4 \\ (u-1)^2 + v^2 = 4 \end{cases}
$$

**Hint from Sauer (EX.2.7.3):**

**b.** The curves are ellipses with semimajor axes 1 and 2 centered at zero and aligned with the $x$ and $y$ axes. Solving by substitution gives the four solutions:

$$
\begin{pmatrix} u_1 \\ v_1 \end{pmatrix} = \begin{pmatrix} \frac{2}{\sqrt{5}} \\ \frac{2}{\sqrt{5}} \end{pmatrix}, \quad
\begin{pmatrix} u_2 \\ v_2 \end{pmatrix} = \begin{pmatrix} -\frac{2}{\sqrt{5}} \\ \frac{2}{\sqrt{5}} \end{pmatrix}, \quad
\begin{pmatrix} u_3 \\ v_3 \end{pmatrix} = \begin{pmatrix} \frac{2}{\sqrt{5}} \\ -\frac{2}{\sqrt{5}} \end{pmatrix}, \quad
\begin{pmatrix} u_4 \\ v_4 \end{pmatrix} = \begin{pmatrix} -\frac{2}{\sqrt{5}} \\ -\frac{2}{\sqrt{5}} \end{pmatrix}
$$

**c.** The curves are a hyperbola and a circle that intersects one half of the hyperbola in two points. Solving by substitution gives the two solutions:

$$
\begin{pmatrix} u_1 \\ v_1 \end{pmatrix} = \begin{pmatrix} \frac{4}{5}(1 + \sqrt{6}) \\ \frac{1}{5}\sqrt{3 + 8\sqrt{6}} \end{pmatrix}, \quad
\begin{pmatrix} u_2 \\ v_2 \end{pmatrix} = \begin{pmatrix} \frac{4}{5}(1 + \sqrt{6}) \\ -\frac{1}{5}\sqrt{3 + 8\sqrt{6}} \end{pmatrix}
$$

**Special Instructions:**

a. First compute the solutions using either Sauer's exact solutions (see Hint above) or `scipy.optimize.fsolve` / `scipy.optimize.root`. This will be useful to compute the forward error.

b. Start Newton's method from a relative distance (measured by the infinity norm) of at least $0.1$ from the solution.

c. At each iteration of Newton's method, you must print:
  - (a) $k$, the iteration number
  - (b) the absolute backward error at iteration $k$, defined by $\|F(x_k)\|_\infty$
  - (c) the relative forward error at iteration $k$, defined by $\|x_k - x\|_\infty / \|x\|_\infty$, where $x$ is the solution computed by `scipy.optimize.fsolve` or `scipy.optimize.root`.

  You may also print the current iterate $x_k$ if you wish.

In [14]:

# part b
def F_b(x):
    u, v = x
    return np.array([u**2 + 4*v**2 - 4,
                      4*u**2 + v**2 - 4])

def J_b(x):
    u, v = x
    return np.array([[2*u, 8*v],
                      [8*u, 2*v]])

def newtons_method(F, J, x0, x_true, tol=1e-12, max_iter=50, label=""):
    x = np.array(x0, dtype=float)
    print(f"--- Newton's method, starting point {label}: x0 = {x} ---")
    for k in range(max_iter):
        Fx = F(x)
        backward_err = np.linalg.norm(Fx, ord=np.inf)
        forward_err = np.linalg.norm(x - x_true, ord=np.inf) / np.linalg.norm(x_true, ord=np.inf)
        print(f"k={k:2d}   x_k = {x}   ||F(x_k)||_inf = {backward_err:.6e}   "
              f"rel. forward err = {forward_err:.6e}")
        if backward_err < tol:
            break
        Jx = J(x)
        delta = np.linalg.solve(Jx, -Fx)
        x = x + delta
    print(f"Converged solution: {x}\n")
    return x


exact_solutions_b = {
    "u1,v1 (++)": np.array([2/np.sqrt(5),  2/np.sqrt(5)]),
    "u2,v2 (-+)": np.array([-2/np.sqrt(5), 2/np.sqrt(5)]),
    "u3,v3 (+-)": np.array([2/np.sqrt(5), -2/np.sqrt(5)]),
    "u4,v4 (--)": np.array([-2/np.sqrt(5),-2/np.sqrt(5)]),
}

print("Exact solutions (Sauer EX.2.7.3, part b):")
for label, sol in exact_solutions_b.items():
    print(f"  {label}: {sol}")
print()

print("Confirming with scipy.optimize.fsolve:")
starting_guesses = {
    "u1,v1 (++)": [1.0, 1.0],
    "u2,v2 (-+)": [-1.0, 1.0],
    "u3,v3 (+-)": [1.0, -1.0],
    "u4,v4 (--)": [-1.0, -1.0],
}
for label, guess in starting_guesses.items():
    sol = fsolve(F_b, guess)
    print(f"  {label}: fsolve -> {sol}   (exact: {exact_solutions_b[label]})")
print()

sol1 = exact_solutions_b["u1,v1 (++)"]
sol2 = exact_solutions_b["u2,v2 (-+)"]
sol3 = exact_solutions_b["u3,v3 (+-)"]
sol4 = exact_solutions_b["u4,v4 (--)"]

start1 = sol1 * 1.2
start2 = sol2 * 1.2
start3 = sol3 * 1.2
start4 = sol4 * 1.2

for label, start, true_sol in [("sol1", start1, sol1), ("sol2", start2, sol2),
                                 ("sol3", start3, sol3), ("sol4", start4, sol4)]:
    rel_dist = np.linalg.norm(start - true_sol, ord=np.inf) / np.linalg.norm(true_sol, ord=np.inf)
    print(f"Relative distance of starting point for {label}: {rel_dist:.4f} (must be >= 0.1)")
print()

newtons_method(F_b, J_b, start1, sol1, label="near (+,+) solution")
newtons_method(F_b, J_b, start2, sol2, label="near (-,+) solution")
newtons_method(F_b, J_b, start3, sol3, label="near (+,-) solution")
newtons_method(F_b, J_b, start4, sol4, label="near (-,-) solution")

Exact solutions (Sauer EX.2.7.3, part b):
  u1,v1 (++): [0.89442719 0.89442719]
  u2,v2 (-+): [-0.89442719  0.89442719]
  u3,v3 (+-): [ 0.89442719 -0.89442719]
  u4,v4 (--): [-0.89442719 -0.89442719]

Confirming with scipy.optimize.fsolve:
  u1,v1 (++): fsolve -> [0.89442719 0.89442719]   (exact: [0.89442719 0.89442719])
  u2,v2 (-+): fsolve -> [-0.89442719  0.89442719]   (exact: [-0.89442719  0.89442719])
  u3,v3 (+-): fsolve -> [ 0.89442719 -0.89442719]   (exact: [ 0.89442719 -0.89442719])
  u4,v4 (--): fsolve -> [-0.89442719 -0.89442719]   (exact: [-0.89442719 -0.89442719])

Relative distance of starting point for sol1: 0.2000 (must be >= 0.1)
Relative distance of starting point for sol2: 0.2000 (must be >= 0.1)
Relative distance of starting point for sol3: 0.2000 (must be >= 0.1)
Relative distance of starting point for sol4: 0.2000 (must be >= 0.1)

--- Newton's method, starting point near (+,+) solution: x0 = [1.07331263 1.07331263] ---
k= 0   x_k = [1.07331263 1.07331263]   ||F(x

array([-0.89442719, -0.89442719])

In [15]:
# part c
def F_c(x):
    u, v = x
    return np.array([u**2 - 4*v**2 - 4,
                      (u - 1)**2 + v**2 - 4])

def J_c(x):
    u, v = x
    return np.array([[2*u,       -8*v],
                      [2*(u - 1), 2*v]])

def newtons_method(F, J, x0, x_true, tol=1e-12, max_iter=50, label=""):
    x = np.array(x0, dtype=float)
    print(f"--- Newton's method, starting point {label}: x0 = {x} ---")
    for k in range(max_iter):
        Fx = F(x)
        backward_err = np.linalg.norm(Fx, ord=np.inf)
        forward_err = np.linalg.norm(x - x_true, ord=np.inf) / np.linalg.norm(x_true, ord=np.inf)
        print(f"k={k:2d}   x_k = {x}   ||F(x_k)||_inf = {backward_err:.6e}   "
              f"rel. forward err = {forward_err:.6e}")
        if backward_err < tol:
            break
        Jx = J(x)
        delta = np.linalg.solve(Jx, -Fx)
        x = x + delta
    print(f"Converged solution: {x}\n")
    return x


u_exact = (4/5) * (1 + np.sqrt(6))
v_exact_mag = (1/5) * np.sqrt(3 + 8*np.sqrt(6))

exact_solutions_c = {
    "u1,v1 (+)": np.array([u_exact,  v_exact_mag]),
    "u2,v2 (-)": np.array([u_exact, -v_exact_mag]),
}

print("Exact solutions (Sauer EX.2.7.3, part c):")
for label, sol in exact_solutions_c.items():
    print(f"  {label}: {sol}")
print()

print("Confirming with scipy.optimize.fsolve:")
starting_guesses = {
    "u1,v1 (+)": [4.0, 2.0],
    "u2,v2 (-)": [4.0, -2.0],
}
for label, guess in starting_guesses.items():
    sol = fsolve(F_c, guess)
    print(f"  {label}: fsolve -> {sol}   (exact: {exact_solutions_c[label]})")
print()

sol1 = exact_solutions_c["u1,v1 (+)"]
sol2 = exact_solutions_c["u2,v2 (-)"]

start1 = sol1 * 1.2
start2 = sol2 * 1.2

for label, start, true_sol in [("sol1", start1, sol1), ("sol2", start2, sol2)]:
    rel_dist = np.linalg.norm(start - true_sol, ord=np.inf) / np.linalg.norm(true_sol, ord=np.inf)
    print(f"Relative distance of starting point for {label}: {rel_dist:.4f} (must be >= 0.1)")
print()

newtons_method(F_c, J_c, start1, sol1, label="near (+) solution")
newtons_method(F_c, J_c, start2, sol2, label="near (-) solution")

Exact solutions (Sauer EX.2.7.3, part c):
  u1,v1 (+): [2.75959179 0.95070328]
  u2,v2 (-): [ 2.75959179 -0.95070328]

Confirming with scipy.optimize.fsolve:
  u1,v1 (+): fsolve -> [2.75959179 0.95070328]   (exact: [2.75959179 0.95070328])
  u2,v2 (-): fsolve -> [ 2.75959179 -0.95070328]   (exact: [ 2.75959179 -0.95070328])

Relative distance of starting point for sol1: 0.2000 (must be >= 0.1)
Relative distance of starting point for sol2: 0.2000 (must be >= 0.1)

--- Newton's method, starting point near (+) solution: x0 = [3.31151015 1.14084393] ---
k= 0   x_k = [3.31151015 1.14084393]   ||F(x_k)||_inf = 2.644604e+00   rel. forward err = 2.000000e-01
k= 1   x_k = [2.82023536 0.97717968]   ||F(x_k)||_inf = 2.681369e-01   rel. forward err = 2.197556e-02
k= 2   x_k = [2.760502   0.95124825]   ||F(x_k)||_inf = 4.240514e-03   rel. forward err = 3.298319e-04
k= 3   x_k = [2.75959201 0.95070348]   ||F(x_k)||_inf = 1.124863e-06   rel. forward err = 7.656547e-08
k= 4   x_k = [2.75959179 0.95070

array([ 2.75959179, -0.95070328])

## CP.2.7.4, Sauer3

Apply Newton's Method to find both solutions of the system of three equations.

$$
\begin{cases}
2u^2 - 4u + v^2 + 3w^2 + 6w + 2 = 0 \\
u^2 + v^2 - 2v + 2w^2 - 5 = 0 \\
3u^2 - 12u + v^2 + 3w^2 + 8 = 0
\end{cases}
$$

**Special Instructions:**

a. First compute the two solutions using either `scipy.optimize.fsolve` or `scipy.optimize.root`. This will be useful to compute the forward error.

b. Start Newton's method from a relative distance (measured by the infinity norm) of at least $0.1$ from the solution.

c. At each iteration of Newton's method, you must print:
  - (a) $k$, the iteration number
  - (b) the absolute backward error at iteration $k$, defined by $\|F(x_k)\|_\infty$
  - (c) the relative forward error at iteration $k$, defined by $\|x_k - x\|_\infty / \|x\|_\infty$, where $x$ is the solution computed by `scipy.optimize.fsolve` or `scipy.optimize.root`.

  You may also print the current iterate $x_k$ if you wish.

**Hint:** Here are the two roots:

$$
\begin{pmatrix} 2 \\ 1 \\ -1 \end{pmatrix}
\qquad \text{and} \qquad
\begin{pmatrix} 1.0960178410014330 \\ -1.1592471842154839 \\ -0.2611479367011116 \end{pmatrix}
\quad \text{(approximation for the second one)}
$$

**Hint:** Here are some function values and Jacobian values you can check before implementing Newton's method:

$$
F\begin{pmatrix} 1 \\ -1 \\ 2 \end{pmatrix} = \begin{pmatrix} 25 \\ 7 \\ 12 \end{pmatrix}, \qquad
F\begin{pmatrix} 3 \\ 5 \\ -4 \end{pmatrix} = \begin{pmatrix} 57 \\ 51 \\ 72 \end{pmatrix}
$$

$$
DF\begin{pmatrix} 1 \\ -1 \\ 2 \end{pmatrix} = \begin{pmatrix} 0 & -2 & 18 \\ 2 & -4 & 8 \\ -6 & -2 & 12 \end{pmatrix}, \qquad
DF\begin{pmatrix} 3 \\ 5 \\ -4 \end{pmatrix} = \begin{pmatrix} 8 & 10 & -18 \\ 6 & 8 & -16 \\ 6 & 10 & -24 \end{pmatrix}
$$

In [16]:
def F(x):
    u, v, w = x
    return np.array([
        2*u**2 - 4*u + v**2 + 3*w**2 + 6*w + 2,
        u**2 + v**2 - 2*v + 2*w**2 - 5,
        3*u**2 - 12*u + v**2 + 3*w**2 + 8
    ])

def J(x):
    u, v, w = x
    return np.array([
        [4*u - 4,    2*v,      6*w + 6],
        [2*u,        2*v - 2,  4*w],
        [6*u - 12,   2*v,      6*w]
    ])

check1 = np.array([1, -1, 2])
check2 = np.array([3, 5, -4])

print("Checking F and J against hint values:")
print("F(1,-1,2) =", F(check1), "  (expected [25, 7, 12])")
print("F(3,5,-4) =", F(check2), "  (expected [57, 51, 72])")
print()
print("DF(1,-1,2) =\n", J(check1))
print("DF(3,5,-4) =\n", J(check2))
print()


def newtons_method(F, J, x0, x_true, tol=1e-13, max_iter=50, label=""):
    x = np.array(x0, dtype=float)
    print(f"--- Newton's method, {label}: x0 = {x} ---")
    for k in range(max_iter):
        Fx = F(x)
        backward_err = np.linalg.norm(Fx, ord=np.inf)
        forward_err = np.linalg.norm(x - x_true, ord=np.inf) / np.linalg.norm(x_true, ord=np.inf)
        print(f"k={k:2d}   x_k = {x}")
        print(f"        ||F(x_k)||_inf = {backward_err:.6e}   rel. forward err = {forward_err:.6e}")
        if backward_err < tol:
            break
        Jx = J(x)
        delta = np.linalg.solve(Jx, -Fx)
        x = x + delta
    print(f"Converged solution: {x}\n")
    return x


# step a
guess1 = [2.0, 1.0, -1.0]
guess2 = [1.0, -1.0, -0.5]

sol1 = fsolve(F, guess1)
sol2 = fsolve(F, guess2)

print("Solutions computed via scipy.optimize.fsolve:")
print("  Solution 1:", sol1, "  (expected [2, 1, -1])")
print("  Solution 2:", sol2, "  (expected [1.0960178410014330, -1.1592471842154839, -0.2611479367011116])")
print()

# step b
start1 = sol1 * 1.2
start2 = sol2 * 1.2

for label, start, true_sol in [("sol1", start1, sol1), ("sol2", start2, sol2)]:
    rel_dist = np.linalg.norm(start - true_sol, ord=np.inf) / np.linalg.norm(true_sol, ord=np.inf)
    print(f"Relative distance of starting point for {label}: {rel_dist:.4f} (must be >= 0.1)")
print()

# step c
newtons_method(F, J, start1, sol1, label="near root 1 (2, 1, -1)")
newtons_method(F, J, start2, sol2, label="near root 2 (~1.096, -1.159, -0.261)")

Checking F and J against hint values:
F(1,-1,2) = [25  7 12]   (expected [25, 7, 12])
F(3,5,-4) = [57 51 72]   (expected [57, 51, 72])

DF(1,-1,2) =
 [[ 0 -2 18]
 [ 2 -4  8]
 [-6 -2 12]]
DF(3,5,-4) =
 [[  8  10 -18]
 [  6   8 -16]
 [  6  10 -24]]

Solutions computed via scipy.optimize.fsolve:
  Solution 1: [ 2.  1. -1.]   (expected [2, 1, -1])
  Solution 2: [ 1.09601784 -1.15924718 -0.26114794]   (expected [1.0960178410014330, -1.1592471842154839, -0.2611479367011116])

Relative distance of starting point for sol1: 0.2000 (must be >= 0.1)
Relative distance of starting point for sol2: 0.2000 (must be >= 0.1)

--- Newton's method, near root 1 (2, 1, -1): x0 = [ 2.4  1.2 -1.2] ---
k= 0   x_k = [ 2.4  1.2 -1.2]
        ||F(x_k)||_inf = 2.680000e+00   rel. forward err = 2.000000e-01
k= 1   x_k = [ 2.01223629  1.15485232 -1.03319269]
        ||F(x_k)||_inf = 5.365944e-01   rel. forward err = 7.742616e-02
k= 2   x_k = [ 2.0036328   1.00536458 -1.00243201]
        ||F(x_k)||_inf = 2.540730e-02

array([ 1.09601784, -1.15924718, -0.26114794])

## CP.2.7.5.b, Sauer3

Use multivariate Newton's Method to find the two points in common of the three given spheres in three-dimensional space.

**(b)** Each sphere has radius 5, with centers $(1, -2, 0)$, $(-2, 2, -1)$, and $(4, -2, 3)$.

In [17]:
c1 = np.array([1., -2., 0.])
c2 = np.array([-2., 2., -1.])
c3 = np.array([4., -2., 3.])
r = 5

def F(x):
    return np.array([
        (x[0]-c1[0])**2 + (x[1]-c1[1])**2 + (x[2]-c1[2])**2 - r**2,
        (x[0]-c2[0])**2 + (x[1]-c2[1])**2 + (x[2]-c2[2])**2 - r**2,
        (x[0]-c3[0])**2 + (x[1]-c3[1])**2 + (x[2]-c3[2])**2 - r**2
    ])

def J(x):
    return np.array([
        [2*(x[0]-c1[0]), 2*(x[1]-c1[1]), 2*(x[2]-c1[2])],
        [2*(x[0]-c2[0]), 2*(x[1]-c2[1]), 2*(x[2]-c2[2])],
        [2*(x[0]-c3[0]), 2*(x[1]-c3[1]), 2*(x[2]-c3[2])]
    ])

def newtons_method(F, J, x0, x_true, tol=1e-13, max_iter=50, label=""):
    x = np.array(x0, dtype=float)
    print(f"--- Newton's method, {label}: x0 = {x} ---")
    for k in range(max_iter):
        Fx = F(x)
        backward_err = np.linalg.norm(Fx, ord=np.inf)
        forward_err = np.linalg.norm(x - x_true, ord=np.inf) / np.linalg.norm(x_true, ord=np.inf)
        print(f"k={k:2d}   x_k = {x}")
        print(f"        ||F(x_k)||_inf = {backward_err:.6e}   rel. forward err = {forward_err:.6e}")
        if backward_err < tol:
            break
        Jx = J(x)
        delta = np.linalg.solve(Jx, -Fx)
        x = x + delta
    print(f"Converged solution: {x}\n")
    return x


# step a
sol1 = fsolve(F, [0, 0, 5])
sol2 = fsolve(F, [2, 2, 2])

print("Solutions computed via scipy.optimize.fsolve:")
print("  Solution 1:", sol1)
print("  Solution 2:", sol2)
print()

# step b
start1 = sol1 * 1.2
start2 = sol2 * 1.2

for label, start, true_sol in [("sol1", start1, sol1), ("sol2", start2, sol2)]:
    rel_dist = np.linalg.norm(start - true_sol, ord=np.inf) / np.linalg.norm(true_sol, ord=np.inf)
    print(f"Relative distance of starting point for {label}: {rel_dist:.4f} (must be >= 0.1)")
print()

# step c
newtons_method(F, J, start1, sol1, label="near intersection point 1")
newtons_method(F, J, start2, sol2, label="near intersection point 2")

Solutions computed via scipy.optimize.fsolve:
  Solution 1: [1. 2. 3.]
  Solution 2: [1.88888889 2.44444444 2.11111111]

Relative distance of starting point for sol1: 0.2000 (must be >= 0.1)
Relative distance of starting point for sol2: 0.2000 (must be >= 0.1)

--- Newton's method, near intersection point 1: x0 = [1.2 2.4 3.6] ---
k= 0   x_k = [1.2 2.4 3.6]
        ||F(x_k)||_inf = 7.360000e+00   rel. forward err = 2.000000e-01
k= 1   x_k = [0.76666667 1.88333333 3.23333333]
        ||F(x_k)||_inf = 5.891667e-01   rel. forward err = 7.777778e-02
k= 2   x_k = [0.95983607 1.97991803 3.04016393]
        ||F(x_k)||_inf = 8.395744e-02   rel. forward err = 1.338798e-02
k= 3   x_k = [0.99833562 1.99916781 3.00166438]
        ||F(x_k)||_inf = 3.334986e-03   rel. forward err = 5.547922e-04
k= 4   x_k = [0.9999969  1.99999845 3.0000031 ]
        ||F(x_k)||_inf = 6.209604e-06   rel. forward err = 1.034930e-06
k= 5   x_k = [1. 2. 3.]
        ||F(x_k)||_inf = 2.168932e-11   rel. forward err = 3.614

array([1.88888889, 2.44444444, 2.11111111])

## CP.2.7.7.b2c, Sauer3

Apply Broyden I with starting guesses $x_0 = (1, 1)$ and $A_0 = I$ to the systems in EX.2.7.3. Report the solutions to as much accuracy as possible and the number of steps required.

**(b)**

$$
\begin{cases} u^2 + 4v^2 = 4 \\ 4u^2 + v^2 = 4 \end{cases}
$$

**(c)**

$$
\begin{cases} u^2 - 4v^2 = 4 \\ (u-1)^2 + v^2 = 4 \end{cases}
$$

In [18]:
def broyden1(F, x0, A0, tol=1e-14, max_iter=100, verbose=True):
    x = np.array(x0, dtype=float)
    A = np.array(A0, dtype=float)
    Fx = F(x)

    if verbose:
        print(f"k= 0   x_k = {x}   ||F(x_k)||_inf = {np.linalg.norm(Fx, ord=np.inf):.6e}")

    for k in range(1, max_iter + 1):
        s = np.linalg.solve(A, -Fx)
        x_new = x + s
        Fx_new = F(x_new)

        y = Fx_new - Fx
        As = A @ s
        A = A + np.outer((y - As), s) / (s @ s)

        x, Fx = x_new, Fx_new

        if verbose:
            print(f"k={k:2d}   x_k = {x}   ||F(x_k)||_inf = {np.linalg.norm(Fx, ord=np.inf):.6e}")

        if np.linalg.norm(Fx, ord=np.inf) < tol:
            break

    return x, k

def F_b(x):
    u, v = x
    return np.array([u**2 + 4*v**2 - 4,
                      4*u**2 + v**2 - 4])

print("="*60)
print("Part (b): Broyden I, x0 = (1,1), A0 = I")
print("="*60)
x0 = np.array([1.0, 1.0])
A0 = np.eye(2)
sol_b, steps_b = broyden1(F_b, x0, A0)
print(f"\nSolution: {sol_b}")
print(f"Number of steps required: {steps_b}")
print(f"Exact solution (Sauer EX.2.7.3b): ({2/np.sqrt(5)}, {2/np.sqrt(5)})")
print()

def F_c(x):
    u, v = x
    return np.array([u**2 - 4*v**2 - 4,
                      (u - 1)**2 + v**2 - 4])

print("="*60)
print("Part (c): Broyden I, x0 = (1,1), A0 = I")
print("="*60)
x0 = np.array([1.0, 1.0])
A0 = np.eye(2)
sol_c, steps_c = broyden1(F_c, x0, A0)
print(f"\nSolution: {sol_c}")
print(f"Number of steps required: {steps_c}")

u_exact = (4/5) * (1 + np.sqrt(6))
v_exact = (1/5) * np.sqrt(3 + 8*np.sqrt(6))
print(f"Exact solution (Sauer EX.2.7.3c): ({u_exact}, {v_exact})  or  ({u_exact}, {-v_exact})")

Part (b): Broyden I, x0 = (1,1), A0 = I
k= 0   x_k = [1. 1.]   ||F(x_k)||_inf = 1.000000e+00
k= 1   x_k = [0. 0.]   ||F(x_k)||_inf = 4.000000e+00
k= 2   x_k = [0.8 0.8]   ||F(x_k)||_inf = 8.000000e-01
k= 3   x_k = [1. 1.]   ||F(x_k)||_inf = 1.000000e+00
k= 4   x_k = [0.88888889 0.88888889]   ||F(x_k)||_inf = 4.938272e-02
k= 5   x_k = [0.89411765 0.89411765]   ||F(x_k)||_inf = 2.768166e-03
k= 6   x_k = [0.89442815 0.89442815]   ||F(x_k)||_inf = 8.599857e-06
k= 7   x_k = [0.89442719 0.89442719]   ||F(x_k)||_inf = 1.489836e-09
k= 8   x_k = [0.89442719 0.89442719]   ||F(x_k)||_inf = 9.281464e-12
k= 9   x_k = [0.89442719 0.89442719]   ||F(x_k)||_inf = 5.911005e-11
k=10   x_k = [0.89442719 0.89442719]   ||F(x_k)||_inf = 8.881784e-16

Solution: [0.89442719 0.89442719]
Number of steps required: 10
Exact solution (Sauer EX.2.7.3b): (0.8944271909999159, 0.8944271909999159)

Part (c): Broyden I, x0 = (1,1), A0 = I
k= 0   x_k = [1. 1.]   ||F(x_k)||_inf = 7.000000e+00
k= 1   x_k = [8. 4.]   ||F(x_k

## CP.2.7.8.b2c, Sauer3

Apply Broyden II with starting guesses $x_0 = (1, 1)$ and $B_0 = I$ to the systems in EX.2.7.3. Report the solutions to as much accuracy as possible and the number of steps required.

**(b)**

$$
\begin{cases} u^2 + 4v^2 = 4 \\ 4u^2 + v^2 = 4 \end{cases}
$$

**(c)**

$$
\begin{cases} u^2 - 4v^2 = 4 \\ (u-1)^2 + v^2 = 4 \end{cases}
$$

In [19]:
def broyden2(F, x0, B0, tol=1e-14, max_iter=100, verbose=True):
    x = np.array(x0, dtype=float)
    B = np.array(B0, dtype=float)
    Fx = F(x)

    if verbose:
        print(f"k= 0   x_k = {x}   ||F(x_k)||_inf = {np.linalg.norm(Fx, ord=np.inf):.6e}")

    for k in range(1, max_iter + 1):

        s = -B @ Fx
        x_new = x + s
        Fx_new = F(x_new)

        y = Fx_new - Fx
        By = B @ y
        B = B + np.outer((s - By), y) / (y @ y)

        x, Fx = x_new, Fx_new

        if verbose:
            print(f"k={k:2d}   x_k = {x}   ||F(x_k)||_inf = {np.linalg.norm(Fx, ord=np.inf):.6e}")

        if np.linalg.norm(Fx, ord=np.inf) < tol:
            break

    return x, k

def F_b(x):
    u, v = x
    return np.array([u**2 + 4*v**2 - 4,
                      4*u**2 + v**2 - 4])

print("="*60)
print("Part (b): Broyden II, x0 = (1,1), B0 = I")
print("="*60)
x0 = np.array([1.0, 1.0])
B0 = np.eye(2)
sol_b, steps_b = broyden2(F_b, x0, B0)
print(f"\nSolution: {sol_b}")
print(f"Number of steps required: {steps_b}")
print(f"Exact solution (Sauer EX.2.7.3b): ({2/np.sqrt(5)}, {2/np.sqrt(5)})")
print()

def F_c(x):
    u, v = x
    return np.array([u**2 - 4*v**2 - 4,
                      (u - 1)**2 + v**2 - 4])

print("="*60)
print("Part (c): Broyden II, x0 = (1,1), B0 = I")
print("="*60)
x0 = np.array([1.0, 1.0])
B0 = np.eye(2)
sol_c, steps_c = broyden2(F_c, x0, B0)
print(f"\nSolution: {sol_c}")
print(f"Number of steps required: {steps_c}")

u_exact = (4/5) * (1 + np.sqrt(6))
v_exact = (1/5) * np.sqrt(3 + 8*np.sqrt(6))
print(f"Exact solution (Sauer EX.2.7.3c): ({u_exact}, {v_exact})  or  ({u_exact}, {-v_exact})")

Part (b): Broyden II, x0 = (1,1), B0 = I
k= 0   x_k = [1. 1.]   ||F(x_k)||_inf = 1.000000e+00
k= 1   x_k = [0. 0.]   ||F(x_k)||_inf = 4.000000e+00
k= 2   x_k = [0.8 0.8]   ||F(x_k)||_inf = 8.000000e-01
k= 3   x_k = [1. 1.]   ||F(x_k)||_inf = 1.000000e+00
k= 4   x_k = [0.88888889 0.88888889]   ||F(x_k)||_inf = 4.938272e-02
k= 5   x_k = [0.89411765 0.89411765]   ||F(x_k)||_inf = 2.768166e-03
k= 6   x_k = [0.89442815 0.89442815]   ||F(x_k)||_inf = 8.599857e-06
k= 7   x_k = [0.89442719 0.89442719]   ||F(x_k)||_inf = 1.488444e-09
k= 8   x_k = [0.89442719 0.89442719]   ||F(x_k)||_inf = 4.152234e-13
k= 9   x_k = [0.89442719 0.89442719]   ||F(x_k)||_inf = 2.636558e-12
k=10   x_k = [0.89442719 0.89442719]   ||F(x_k)||_inf = 4.440892e-16

Solution: [0.89442719 0.89442719]
Number of steps required: 10
Exact solution (Sauer EX.2.7.3b): (0.8944271909999159, 0.8944271909999159)

Part (c): Broyden II, x0 = (1,1), B0 = I
k= 0   x_k = [1. 1.]   ||F(x_k)||_inf = 7.000000e+00
k= 1   x_k = [8. 4.]   ||F(x